### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b", max_tokens=800)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1'}}, client=<groq.resources.chat.completions.Completions object at 0x115f5f9e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x116c32e70>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=800)

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain.messages import HumanMessage,SystemMessage

In [10]:
from langchain_groq import ChatGroq

In [11]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    max_tokens=500 
)

In [12]:
agent = create_agent(
    model = llm,
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:qwen/qwen3.8-27b",
            trigger=("messages",10),
            keep = ("messages",4)
        )
    ]
)

In [13]:
config={"configurable":{"thread_id":"test-1"}}

In [14]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='4fcace9f-1452-42fe-8f11-1597f929b461'), AIMessage(content='2 + 2 = **4**', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 19, 'total_tokens': 28, 'completion_time': 0.017285822, 'completion_tokens_details': None, 'prompt_time': 0.001013214, 'prompt_tokens_details': None, 'queue_time': 0.004101597, 'total_time': 0.018299036}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e1d4-ac62-7cf2-9dd2-a02b80fd2f18-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 9, 'total_tokens': 28})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='4fcace9f-1452-42fe-8f11-1597f929b461'), AIMessage(

## token Size

In [23]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

# Use the GPT-OSS model which correctly supports tool calling in your environment
llm = ChatGroq(
    model="openai/gpt-oss-20b", 
    max_tokens=500 
)

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm, # Use the working model here too
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token.    # Run test

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")



Paris: ~284 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='f9daef71-14be-49a0-a505-0e5b9c60c2e8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says: "Find hotels in Paris". We need to use the function search_hotels with city "Paris". We should call the function and then respond. The function returns a long response. We should use that.', 'tool_calls': [{'id': 'fc_5194f27b-44e4-4c14-ba33-54a5e9a6e0d8', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 129, 'total_tokens': 199, 'completion_time': 0.076008225, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.006235754, 'prompt_tokens_details': None, 'queue_time': 0.195426787, 'total_time': 0.082243979}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e2cb7a84ec', 'service_tier': 'on_demand'